# Env

In [ ]:
import os

import numpy as np
import torch
import torch.nn.functional as F

from transformers import (T5TokenizerFast,
                          AutoConfig,
                          GenerationConfig,
                          T5ForConditionalGeneration,
                          Seq2SeqTrainer,
                          Seq2SeqTrainingArguments)

In [ ]:
# work dir
work_dir = '/home/ubuntu/nlp-practice'

In [ ]:
%cd {work_dir}
!pwd

In [ ]:
# tokeinzer warning disable
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Seq2Seq + Attention

In [ ]:
# Gradient False
torch.set_grad_enabled(False)

In [ ]:
tokenizer = T5TokenizerFast.from_pretrained('data/aihub_koen_32k')

In [ ]:
bos = "<s>"
eos = "</s>"
src_data = [
            "나는 학생입니다.",
            "나는 학교에 가는 것을 좋아합니다."
        ]
tgt_data = [
            "I am a student.",
            "I love to go to school."
        ]

In [ ]:
enc_inputs = src_data
enc_inputs

In [ ]:
enc_x = tokenizer(enc_inputs, return_tensors="pt", padding=True)
enc_x

In [ ]:
dec_inputs = [f"{bos}{text}" for text in tgt_data]
dec_inputs

In [ ]:
dec_x = tokenizer(dec_inputs, return_tensors="pt", padding=True)
dec_x

In [ ]:
dec_labels = [f"{text}{eos}" for text in tgt_data]
dec_labels

In [ ]:
dec_y = tokenizer(dec_labels, return_tensors="pt", padding=True)
dec_y

## Seq2Seq

In [ ]:
n_layers = 2
hidden_dim = 4
vocab_size = tokenizer.vocab_size
pad_idx = tokenizer.pad_token_id

In [ ]:
embedding = torch.nn.Embedding(vocab_size, hidden_dim, padding_idx=pad_idx)

encoder = torch.nn.RNN(
    hidden_dim,
    hidden_dim // 2,
    num_layers=n_layers,
    bidirectional=True,  # bidirectional=True for Encoder
    dropout=0.1,
    batch_first=True,  # If False, input shape is (seq_len, batch_size, input_size).
)

decoder = torch.nn.RNN(
    hidden_dim,
    hidden_dim,  # encoder bidirectional=True, decode bidirectional=False
    num_layers=n_layers,
    bidirectional=False,  # bidirectional=False for Decoder (LM)
    dropout=0.1,
    batch_first=True,  # If False, input shape is (seq_len, batch_size, input_size).
)

# Note that we use "vocab_size" sence we are prediting vocab.
fc = torch.nn.Linear(hidden_dim, vocab_size)

### encoder

In [ ]:
enc_embed = embedding(enc_x['input_ids'])
 # |enc_embed| = (batch_size, seq_len_enc, embedding_dim)
enc_embed

In [ ]:
enc_out, hidden_e = encoder(enc_embed)
# |enc_out| = (batch_size, seq_len_enc, hidden_dim)
# |hidden| = (n_layers * 2, batch_size, hidden_dim // 2)
enc_out, hidden_e

In [ ]:
hidden = torch.cat((hidden_e[0::2], hidden_e[1::2]), dim=-1)
# |hidden| = (n_layers, batch_size, hidden_dim)
hidden

### decoder

In [ ]:
dec_embed = embedding(dec_x['input_ids'])
dec_embed

In [ ]:
dec_out, hidden = decoder(dec_embed, hidden)
# |dec_out| = (batch_size, seq_len_dec, hidden_dim)
# |hidden| = (n_layers, batch_size, hidden_dim)
dec_out, hidden

### linear & softmax

In [ ]:
hidden = dec_out

In [ ]:
logits = fc(hidden)
# |logits| = (batch_size, seq_len_dec, vocab_size)
logits.shape

### loss

In [ ]:
criterion = torch.nn.CrossEntropyLoss()

In [ ]:
labels_id = dec_y['input_ids']
labels_id

In [ ]:
loss = criterion(logits.view(-1, logits.size(-1)), labels_id.view(-1,))
loss

## Seq2Seq + Attention

In [ ]:
n_layers = 2
hidden_dim = 4
vocab_size = tokenizer.vocab_size
pad_idx = tokenizer.pad_token_id

In [ ]:
embedding = torch.nn.Embedding(vocab_size, hidden_dim, padding_idx=pad_idx)

encoder = torch.nn.RNN(
    hidden_dim,
    hidden_dim // 2,
    num_layers=n_layers,
    bidirectional=True,  # bidirectional=True for Encoder
    dropout=0.1,
    batch_first=True,  # If False, input shape is (seq_len, batch_size, input_size).
)

decoder = torch.nn.RNN(
    hidden_dim,
    hidden_dim,  # encoder bidirectional=True, decode bidirectional=False
    num_layers=n_layers,
    bidirectional=False,  # bidirectional=False for Decoder (LM)
    dropout=0.1,
    batch_first=True,  # If False, input shape is (seq_len, batch_size, input_size).
)

# attention weights
attn_w = torch.nn.Linear(hidden_dim, hidden_dim)
concat_w = torch.nn.Linear(hidden_dim * 2, hidden_dim)

# Note that we use "vocab_size" sence we are prediting vocab.
fc = torch.nn.Linear(hidden_dim, vocab_size)

### encoder

In [ ]:
enc_embed = embedding(enc_x['input_ids'])
 # |enc_embed| = (batch_size, seq_len_enc, embedding_dim)
enc_embed

In [ ]:
enc_out, hidden_e = encoder(enc_embed)
# |enc_out| = (batch_size, seq_len_enc, hidden_dim)
# |hidden| = (n_layers * 2, batch_size, hidden_dim // 2)
enc_out, hidden_e

In [ ]:
hidden = torch.cat((hidden_e[0::2], hidden_e[1::2]), dim=-1)
# |hidden| = (n_layers, batch_size, hidden_dim)
hidden

### decoder

In [ ]:
dec_embed = embedding(dec_x['input_ids'])
dec_embed

In [ ]:
dec_out, hidden = decoder(dec_embed, hidden)
# |dec_out| = (batch_size, seq_len_dec, hidden_dim)
# |hidden| = (n_layers, batch_size, hidden_dim)
dec_out, hidden

### attention

In [ ]:
Q = dec_out
K = enc_out
V = enc_out
attention_mask = enc_x['attention_mask']
# |Q| = (batch_size, Q_len, hidden_dim)
# |K| = (batch_size, K_len, hidden_dim)
# |V| = (batch_size, K_len, hidden_dim)
# |attention_mask| = (batch_size, K_len)

In [ ]:
Q = attn_w(Q)
# |Q| = (batch_size, Q_len, hidden_dim)
Q

In [ ]:
attn_score = torch.matmul(Q, K.transpose(-2, -1).contiguous())
# |attn_score| = (batch_size, Q_len, K_len)
attn_score

In [ ]:
attention_mask = attention_mask.unsqueeze(1)
# |attention_mask| = (batch_size, 1, K_len)
attention_mask

In [ ]:
attn_score -= (1 - attention_mask) * 1e9
# |attn_score| = (batch_size, Q_len, K_len)
attn_score

In [ ]:
attn_prob = F.softmax(attn_score, dim=-1)
# |attn_prob| = (batch_size, Q_len, K_len)
attn_prob

In [ ]:
attn_out = torch.matmul(attn_prob, V)
# |attn_out| = (batch_size, Q_len, hidden_dim)
attn_out

In [ ]:
hidden = torch.cat([Q, attn_out], dim=-1)
# |hidden| = (batch_size, Q_len, hidden_dim * 2)
hidden = concat_w(hidden)
hidden = F.tanh(hidden)
# |hidden| = (batch_size, Q_len, hidden_dim)
hidden

### linear & softmax

In [ ]:
logits = fc(hidden)
# |logits| = (batch_size, seq_len_dec, vocab_size)
logits.shape

### loss

In [ ]:
criterion = torch.nn.CrossEntropyLoss()

In [ ]:
labels_id = dec_y['input_ids']
labels_id

In [ ]:
loss = criterion(logits.view(-1, logits.size(-1)), labels_id.view(-1,))
loss

# Transformer

## Tutorial

In [ ]:
# Gradient False
torch.set_grad_enabled(False)

In [ ]:
tokenizer = T5TokenizerFast.from_pretrained('data/aihub_koen_32k')

### Data

In [ ]:
bos = "<s>"
eos = "</s>"
src_data = [
            "나는 학생입니다.",
            "나는 학교에 가는 것을 좋아합니다."
        ]
tgt_data = [
            "I am a student.",
            "I love to go to school."
        ]

In [ ]:
enc_inputs = src_data
enc_inputs

In [ ]:
enc_x = tokenizer(enc_inputs, return_tensors="pt", padding=True)
enc_x

In [ ]:
dec_inputs = [f"{bos}{text}" for text in tgt_data]
dec_inputs

In [ ]:
dec_x = tokenizer(dec_inputs, return_tensors="pt", padding=True)
dec_x

In [ ]:
dec_labels = [f"{text}{eos}" for text in tgt_data]
dec_labels

In [ ]:
dec_y = tokenizer(dec_labels, return_tensors="pt", padding=True)
dec_y

### Models

In [ ]:
hidden_dim = 4
vocab_size = tokenizer.vocab_size
pad_idx = tokenizer.pad_token_id

n_head = 2
d_head = hidden_dim // n_head

In [ ]:
embedding = torch.nn.Embedding(vocab_size, hidden_dim, padding_idx=pad_idx)

W_Q = torch.nn.Linear(hidden_dim, n_head * d_head)
W_K = torch.nn.Linear(hidden_dim, n_head * d_head)
W_V = torch.nn.Linear(hidden_dim, n_head * d_head)
W_O = torch.nn.Linear(n_head * d_head, hidden_dim)

### Mask

In [ ]:
enc_mask = enc_x['attention_mask'].unsqueeze(1)
enc_mask

In [ ]:
dec_len = dec_x['attention_mask'].shape[1]
dec_mask = torch.ones(dec_len, dec_len)
dec_mask = 1 - dec_mask.triu(diagonal=1)
dec_mask = dec_mask.unsqueeze(0)
dec_mask

### Embedding

In [ ]:
enc_hidden = embedding(enc_x['input_ids'])
dec_hidden = embedding(dec_x['input_ids'])
enc_hidden, dec_hidden

### scale-dot product attention

In [ ]:
def scale_dot_product_attention(Q, K, V, attention_mask):
    # |Q| = (batch_size, n_head, Q_len, hidden_dim)
    # |K| = (batch_size, n_head, K_len, hidden_dim)
    # |V| = (batch_size, n_head, K_len, hidden_dim)
    # |attention_mask| = (batch_size, 1, 1 or K_len, K_len)

    # d_k
    d_k = torch.tensor(K.shape[-1], dtype=K.dtype, device=K.device)
    scale = torch.sqrt(d_k) # scalar
    # |Q| = (batch_size, n_head, Q_len, hidden_dim)
    attn_score = torch.matmul(Q, K.transpose(-2, -1).contiguous())
    attn_score = attn_score.div(scale)
    attn_score -= (1 - attention_mask) * 1e9
    # |attn_score| = (batch_size, n_head, Q_len, K_len)
    attn_prob = F.softmax(attn_score, dim=-1)
    # |attn_prob| = (batch_size, n_head, Q_len, K_len)
    attn_out = torch.matmul(attn_prob, V)
    # |attn_out| = (batch_size, n_head, Q_len, hidden_dim)
    return attn_out

In [ ]:
# encoder self attention
Q = enc_hidden
K = enc_hidden
V = enc_hidden
attention_mask = enc_mask

scale_dot_product_attention(Q, K, V, attention_mask)

In [ ]:
# decoder self attention
Q = dec_hidden
K = dec_hidden
V = dec_hidden
attention_mask = dec_mask

scale_dot_product_attention(Q, K, V, attention_mask)

In [ ]:
# cross attention
Q = dec_hidden
K = enc_hidden
V = enc_hidden
attention_mask = enc_mask

scale_dot_product_attention(Q, K, V, attention_mask)

### multi head attention

In [ ]:
def multi_head_product_attention(Q, K, V, attention_mask):
    # |Q| = (batch_size, Q_len, hidden_dim)
    # |K| = (batch_size, K_len, hidden_dim)
    # |V| = (batch_size, K_len, hidden_dim)
    # |attention_mask| = (batch_size, 1 or Q_len, K_len)

    Q_m = W_Q(Q).view(-1, Q.size(1), n_head, d_head).transpose(1, 2).contiguous()
    K_m = W_K(K).view(-1, K.size(1), n_head, d_head).transpose(1, 2).contiguous()
    V_m = W_V(V).view(-1, V.size(1), n_head, d_head).transpose(1, 2).contiguous()
    # |Q_m| = (batch_size, n_head, Q_len, d_head)
    # |K_m| = (batch_size, n_head, K_len, d_head)
    # |V_m| = (batch_size, n_head, K_len, d_head)

    attention_mask_m = attention_mask.unsqueeze(1)
    # |attention_mask| = (batch_size, 1, 1 or Q_len, K_len)

    attn_out_m = scale_dot_product_attention(Q_m, K_m, V_m, attention_mask_m)
    # |attn_out_m| = (batch_size, n_head, Q_len, d_head)

    attn_out_c = attn_out_m.transpose(1, 2).contiguous().view(-1, Q.size(1), n_head * d_head)
    # |attn_out_c| = (batch_size, Q_len, n_head * d_head)

    attn_out = W_O(attn_out_c)
    # |attn_out_c| = (batch_size, Q_len, hidden_dim)

    return attn_out

In [ ]:
# encoder self attention
Q = enc_hidden
K = enc_hidden
V = enc_hidden
attention_mask = enc_mask

multi_head_product_attention(Q, K, V, attention_mask)

In [ ]:
# decoder self attention
Q = dec_hidden
K = dec_hidden
V = dec_hidden
attention_mask = dec_mask

multi_head_product_attention(Q, K, V, attention_mask)

In [ ]:
# cross attention
Q = dec_hidden
K = enc_hidden
V = enc_hidden
attention_mask = enc_mask

scale_dot_product_attention(Q, K, V, attention_mask)

## Train

In [ ]:
# !python train_transformer.py

## Test

In [ ]:
!ls results/t5-nmt/

In [ ]:
device = torch.device("cpu")

In [ ]:
model_fn = "results/t5-nmt/checkpoint-67602"

tokenizer = T5TokenizerFast.from_pretrained(model_fn)

model = T5ForConditionalGeneration.from_pretrained(model_fn)
model = model.to(device)
model.eval()

In [ ]:
generation_config = GenerationConfig(
        max_new_tokens=512,
        early_stopping=True,
        do_sample=False,
        num_beams=8,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        decoder_start_token_id=tokenizer.bos_token_id,
        repetition_penalty=1.2,
        length_penalty=1.0,
    )

In [ ]:
from train_transformer import NMTDataset

# valid dataset
valid_dataset = NMTDataset("data/aihub_koen/valid.ko", "data/aihub_koen/valid.en")

In [ ]:
src = []
indexs = np.random.randint(0, len(valid_dataset), 10)
for idx in indexs:
    src.append(valid_dataset[idx][0])
src

In [ ]:
input_ids = tokenizer.batch_encode_plus(
        src,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    ).input_ids.to(device)
input_ids

In [ ]:
beam_output = model.generate(
        input_ids=input_ids,
        generation_config=generation_config,
    )

In [ ]:
for line, tgt in zip(src, beam_output):
    result = tokenizer.decode(tgt, skip_special_tokens=True)
    print(f"- ko: {line}\n- en: {result}\n")

## Infer

In [ ]:
device = torch.device("cpu")

In [ ]:
# model_fn = "results/t5-nmt/checkpoint-67602"
model_fn = "cchyun/nmt-koen-t5-small"

tokenizer = T5TokenizerFast.from_pretrained(model_fn)

model = T5ForConditionalGeneration.from_pretrained(model_fn)
model = model.to(device)
model.eval()

In [ ]:
generation_config = GenerationConfig(
        max_new_tokens=512,
        early_stopping=True,
        do_sample=False,
        num_beams=8,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        decoder_start_token_id=tokenizer.bos_token_id,
        repetition_penalty=1.2,
        length_penalty=1.0,
    )

In [ ]:
while True:
    print("input> ", end="")
    line = str(input())
    if len(line) == 0:
        break

    x = tokenizer(
        line,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )["input_ids"].to(device)

    beam_output = model.generate(
        input_ids=x,
        generation_config=generation_config,
    )
    result = tokenizer.decode(beam_output[0], skip_special_tokens=True)

    print(f"- ko: {line}\n- en: {result}\n")